 Model 7 — Multimodal Genre Classification


 Imports

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LSTM, GRU, Embedding,
    GlobalAveragePooling2D, BatchNormalization,
    Bidirectional, Concatenate, Lambda
)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from PIL import Image

print(f"✅ Ready! TensorFlow: {tf.__version__}")

 Load Dataset

In [ ]:

CSV_PATH  = r"final_movie_dataset.csv"
IMAGE_DIR = r"dataset_by_genre"

IMG_SIZE   = 224
MAX_WORDS  = 10000
MAX_LEN    = 100
EMBED_DIM  = 128
NUM_EPOCHS = 30
BATCH_SIZE = 32

# Load CSV
df = pd.read_csv(CSV_PATH)
df.columns = ['img_url','img_url2','movie_url','title','title2',
               'year','duration','rating','votes','description',
               'extra1','extra2','genre','image_path']
df = df.dropna(subset=['description','genre','image_path']).reset_index(drop=True)

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['genre'])
NUM_CLASSES = len(le.classes_)
GENRES      = le.classes_.tolist()

print(f"✅ Loaded {len(df)} samples — {NUM_CLASSES} genres")
print(f"Genres: {GENRES}")
print()
print(df['genre'].value_counts())

 Prepare Image Data

In [ ]:
def load_image(genre, img_filename):
    """Load image from dataset_by_genre/GENRE/filename"""
    img_path = os.path.join(IMAGE_DIR, genre, os.path.basename(img_filename))
    try:
        img = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        return img_to_array(img) / 255.0
    except:
        # Return black image if file not found
        return np.zeros((IMG_SIZE, IMG_SIZE, 3))

print("⏳ Loading images... (this may take a minute)")
X_images = np.array([
    load_image(row['genre'], row['image_path'])
    for _, row in df.iterrows()
], dtype='float32')

print(f"✅ Images loaded: {X_images.shape}")

 Prepare Text Data

In [ ]:
texts  = df['description'].astype(str).tolist()
labels = df['label'].tolist()

# Tokenize descriptions
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X_text    = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y_labels  = to_categorical(labels, NUM_CLASSES)

print(f"✅ Text ready: {X_text.shape}")
print(f"Vocab size: {len(tokenizer.word_index)}")

Train / Test Split

In [ ]:
indices = np.arange(len(df))
(
    idx_train, idx_test
) = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels)

X_img_train,  X_img_test  = X_images[idx_train], X_images[idx_test]
X_text_train, X_text_test = X_text[idx_train],   X_text[idx_test]
y_train,      y_test       = y_labels[idx_train], y_labels[idx_test]

print(f"✅ Train: {len(idx_train)} | Test: {len(idx_test)}")

 Build Multimodal Model

In [ ]:
# ════════════════════════════════════
# Branch 1: Image (CNN)
# ════════════════════════════════════
img_input  = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_input')

resnet     = ResNet50(weights='imagenet', include_top=False, input_tensor=img_input)
for layer in resnet.layers:
    layer.trainable = False          # freeze all — fine-tune later
for layer in resnet.layers[-20:]:    # unfreeze last 20 layers
    layer.trainable = True

x_img = GlobalAveragePooling2D()(resnet.output)
x_img = Dense(256, activation='relu')(x_img)
x_img = BatchNormalization()(x_img)
x_img = Dropout(0.4)(x_img)

# ════════════════════════════════════
# Branch 2: Text (BiLSTM)
# ════════════════════════════════════
text_input = Input(shape=(MAX_LEN,), name='text_input')

x_text = Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN)(text_input)
x_text = Bidirectional(LSTM(128, return_sequences=True))(x_text)
x_text = Dropout(0.3)(x_text)
x_text = Bidirectional(LSTM(64))(x_text)
x_text = Dense(256, activation='relu')(x_text)
x_text = Dropout(0.3)(x_text)

# ════════════════════════════════════
# Fusion: Concatenate both branches
# ════════════════════════════════════
fused  = Concatenate(name='fusion')([x_img, x_text])   # 256 + 256 = 512
fused  = Dense(256, activation='relu')(fused)
fused  = BatchNormalization()(fused)
fused  = Dropout(0.4)(fused)
fused  = Dense(128, activation='relu')(fused)
fused  = Dropout(0.3)(fused)
output = Dense(NUM_CLASSES, activation='softmax', name='output')(fused)

# ════════════════════════════════════
# Final Model
# ════════════════════════════════════
multimodal_model = Model(
    inputs  = [img_input, text_input],
    outputs = output,
    name    = 'Multimodal_CNN_LSTM'
)

multimodal_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

multimodal_model.summary()

total_params     = multimodal_model.count_params()
trainable_params = sum([tf.size(v).numpy() for v in multimodal_model.trainable_variables])
print(f"\nTotal params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

 Train Multimodal Model

In [ ]:
# Class weights to handle imbalance (Comedy has fewer samples)
from sklearn.utils.class_weight import compute_class_weight

y_train_int   = np.argmax(y_train, axis=1)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_int),
    y=y_train_int
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:")
for i, g in enumerate(GENRES):
    print(f"  {g:<22}: {class_weight_dict[i]:.3f}")

In [ ]:
callbacks_mm = [
    EarlyStopping(patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=4, verbose=1, min_lr=1e-7)
]

print("🚀 Training Multimodal Model (CNN + LSTM)...")
print("   This uses BOTH image + text simultaneously\n")

history_mm = multimodal_model.fit(
    x          = [X_img_train, X_text_train],   # ← 2 inputs
    y          = y_train,
    validation_data = ([X_img_test, X_text_test], y_test),
    epochs     = NUM_EPOCHS,
    batch_size = BATCH_SIZE,
    class_weight = class_weight_dict,
    callbacks  = callbacks_mm
)

mm_val_acc = max(history_mm.history['val_accuracy'])
print(f"\n✅ Multimodal Best Val Accuracy: {mm_val_acc:.4f} ({mm_val_acc*100:.2f}%)")

 Evaluate Results

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_mm.history['accuracy'],     label='Train', color='#378ADD')
axes[0].plot(history_mm.history['val_accuracy'], label='Val',   color='#D85A30')
axes[0].set_title('Multimodal — Accuracy'); axes[0].legend()
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')

axes[1].plot(history_mm.history['loss'],     label='Train', color='#378ADD')
axes[1].plot(history_mm.history['val_loss'], label='Val',   color='#D85A30')
axes[1].set_title('Multimodal — Loss'); axes[1].legend()
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')

plt.suptitle('Multimodal CNN + LSTM Training', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Predictions
y_pred_probs = multimodal_model.predict([X_img_test, X_text_test], verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = np.argmax(y_test, axis=1)

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=GENRES, yticklabels=GENRES)
plt.title('Multimodal — Confusion Matrix', fontsize=13)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

# Classification Report
print("\n📊 Multimodal Classification Report:")
print(classification_report(y_true, y_pred, target_names=GENRES))

 Compare Multimodal vs Other Models

In [ ]:
# ⚠️ Replace with your actual results from the 6 models notebook
model_names = ['CNN\n(Image)', 'LSTM\n(Text)', 'GRU\n(Text)', 'BERT\n(Text)', 'AutoEncoder', 'Multimodal\n(Image+Text)']
accuracies  = [
    0.45,   # ← replace with your cnn_val_acc
    0.52,   # ← replace with your lstm_val_acc
    0.51,   # ← replace with your gru_val_acc
    0.63,   # ← replace with your bert_val_acc
    0.48,   # ← replace with your ae_val_acc
    mm_val_acc  # ← Multimodal (calculated automatically)
]

colors = ['#D3D1C7','#D3D1C7','#D3D1C7','#D3D1C7','#D3D1C7','#378ADD']

plt.figure(figsize=(11, 5))
bars = plt.bar([m.replace('\n', ' ') for m in model_names],
               [a * 100 for a in accuracies],
               color=colors, edgecolor='white', linewidth=1.2)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc*100:.1f}%', ha='center', va='bottom',
             fontsize=10, fontweight='bold')

plt.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
plt.ylim(0, 100)
plt.title('All Models Comparison — Multimodal is highlighted', fontsize=13)
plt.ylabel('Val Accuracy (%)')
plt.xticks(rotation=15)
plt.legend()
plt.tight_layout()
plt.show()

best_idx = int(np.argmax(accuracies))
print(f"\n🏆 Best Overall: {model_names[best_idx].replace(chr(10), ' ')} — {accuracies[best_idx]*100:.2f}%")

 Give Image + Description

In [ ]:
def predict_multimodal(img_path, description):
    """
    Predict movie genre using BOTH poster image AND description.
    """
    # --- Prepare image ---
    img     = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_arr = img_to_array(img) / 255.0
    img_arr = np.expand_dims(img_arr, axis=0)                       # (1, 224, 224, 3)

    # --- Prepare text ---
    seq     = tokenizer.texts_to_sequences([description])
    padded  = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')  # (1, 100)

    # --- Predict ---
    probs   = multimodal_model.predict([img_arr, padded], verbose=0)[0]
    top_idx = np.argsort(probs)[::-1]

    # --- Visualize ---
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].imshow(load_img(img_path))
    axes[0].set_title('Movie Poster', fontsize=11)
    axes[0].axis('off')

    bar_colors = ['#378ADD' if i == top_idx[0] else '#D3D1C7' for i in range(NUM_CLASSES)]
    axes[1].barh(GENRES, probs * 100, color=bar_colors)
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_xlim(0, 100)
    axes[1].set_title('Multimodal Prediction', fontsize=11)
    for i, p in enumerate(probs):
        axes[1].text(p * 100 + 0.5, i, f'{p*100:.1f}%', va='center', fontsize=9)

    plt.suptitle(
        f'✅ Predicted: {GENRES[top_idx[0]]}  ({probs[top_idx[0]]*100:.1f}%)',
        fontsize=13, fontweight='bold', color='#378ADD'
    )
    plt.tight_layout(); plt.show()

    print(f"\n📝 Description: {description[:80]}...")
    print(f"\n📊 Full breakdown:")
    print(f"{'Genre':<22} {'Confidence':>12}")
    print('─' * 36)
    for i in top_idx:
        bar = '█' * int(probs[i] * 25)
        print(f"{GENRES[i]:<22} {probs[i]*100:>6.2f}%  {bar}")
    print(f"\n🏆 Final Prediction: {GENRES[top_idx[0]]}")
    return GENRES[top_idx[0]]


# ══════════════════════════════════════════
# TEST — غير الـ path والـ description
# ══════════════════════════════════════════
predict_multimodal(
    img_path    = r"dataset_by_genre\Comedy\89.jpg",
    description = "Three friends navigate college life with humor, friendship, and unexpected adventures."
)

 Save the Model

In [ ]:
multimodal_model.save('multimodal_genre_model.keras')
print("✅ Model saved as: multimodal_genre_model.keras")
print("\nTo load it later:")
print("  from tensorflow.keras.models import load_model")
print("  model = load_model('multimodal_genre_model.keras')")